[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pabanib/Clases_clasificacion/blob/main/Práctico_orange.ipynb#copy=true)


[![Open in Anaconda](https://static.anaconda.cloud/content/a22d04e8445b700f28937ab3231b8cded505d0395c63b7a269696722196d5415)](https://anaconda.com/api/nbserve/launch_notebook?nb_url=https%3A%2F%2Fanaconda.com%2Fapi%2Fprojects%2F6905e11d-e422-418c-ad09-bff2871f164e%2Ffiles%2FPráctico_orange.ipynb%3Fversion%3Dac7c43d2-9e40-4aa9-a0a9-0775b55380fe)


# Trabajo práctico clasificación

## Base de datos OJ: compra de jugo de naranja

La base **OJ (Orange Juice)** contiene **1.070 compras y 18 variables**. Cada fila representa una compra de una de dos marcas: **Citrus Hill (CH)** o **Minute Maid (MM)**.

## Situación y objetivo

Sos consultor de **Minute Maid (MM)**, que registra menos compras que Citrus Hill (CH) en esta muestra. La empresa busca comprender la elección de los clientes y explorar oportunidades para aumentar sus ventas.

Tu tarea es desarrollar un modelo de clasificación que permita predecir la marca comprada e identificar las variables que más contribuyen a esa predicción. Considerá a **MM como la clase positiva**.

A partir de los resultados, elaborá una recomendación para la empresa: ¿qué factores se relacionan con la elección de MM? ¿Cuáles podría modificar la marca? ¿Qué acciones propondrías y qué evidencia adicional necesitarías para comprobar que mejoran las ventas?

Predecir qué marca comprará un cliente según su fidelidad, los precios y las promociones. Es un problema de **clasificación binaria**, con `Purchase` como variable respuesta. Utiliza las herramientas de clasificación vistas en clases.




## Diccionario de variables

| Variable | Descripción |
|---|---|
| `Purchase` | Marca comprada: `CH` o `MM`. |
| `WeekofPurchase` | Semana de compra. |
| `StoreID` | Identificador del comercio. |
| `PriceCH` | Precio de lista de CH. |
| `PriceMM` | Precio de lista de MM. |
| `DiscCH` | Descuento monetario de CH. |
| `DiscMM` | Descuento monetario de MM. |
| `SpecialCH` | Indicador de promoción especial de CH. |
| `SpecialMM` | Indicador de promoción especial de MM. |
| `LoyalCH` | Medida de fidelidad del cliente a CH. |
| `SalePriceMM` | Precio de MM después del descuento. |
| `SalePriceCH` | Precio de CH después del descuento. |
| `PriceDiff` | Diferencia: `SalePriceMM − SalePriceCH`. |
| `Store7` | Indica si la compra ocurrió en el comercio 7. |
| `PctDiscMM` | Descuento relativo de MM. |
| `PctDiscCH` | Descuento relativo de CH. |
| `ListPriceDiff` | Diferencia: `PriceMM − PriceCH`. |
| `STORE` | Comercio de compra, entre cinco posibles. |

Fuente: [Documentación oficial de ISLP](https://intro-stat-learning.github.io/ISLP/datasets/OJ.html).

## Consideraciones para el análisis

- Un `PriceDiff` positivo indica que MM es más caro que CH después de los descuentos.
- Los identificadores de comercio deben interpretarse como categorías, aunque estén representados mediante números.
- La importancia de una variable para predecir la compra no demuestra que modificarla cause un cambio en la elección del cliente.



In [ ]:
import pandas as pd
import numpy as np

url  = url = "https://raw.githubusercontent.com/pabanib/dataframes/master/orange_juice.csv"

oj = pd.read_csv(url)
oj.head()

,Purchase,WeekofPurchase,StoreID,PriceCH,PriceMM,DiscCH,DiscMM,SpecialCH,SpecialMM,LoyalCH,SalePriceMM,SalePriceCH,PriceDiff,Store7,PctDiscMM,PctDiscCH,ListPriceDiff,STORE
0,CH,237,1,1.75,1.99,0.00,0.0,0,0,0.500000,1.99,1.75,0.24,No,0.000000,0.000000,0.24,1
1,CH,239,1,1.75,1.99,0.00,0.3,0,1,0.600000,1.69,1.75,-0.06,No,0.150754,0.000000,0.24,1
2,CH,245,1,1.86,2.09,0.17,0.0,0,0,0.680000,2.09,1.69,0.40,No,0.000000,0.091398,0.23,1
3,MM,227,1,1.69,1.69,0.00,0.0,0,0,0.400000,1.69,1.69,0.00,No,0.000000,0.000000,0.00,1
4,CH,228,7,1.69,1.69,0.00,0.0,0,0,0.956535,1.69,1.69,0.00,Yes,0.000000,0.000000,0.00,0


In [3]:
oj["Purchase"].value_counts()

Purchase
CH    653
MM    417
Name: count, dtype: int64

In [4]:
oj.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1070 entries, 0 to 1069
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   Purchase        1070 non-null   category
 1   WeekofPurchase  1070 non-null   int64   
 2   StoreID         1070 non-null   int64   
 3   PriceCH         1070 non-null   float64 
 4   PriceMM         1070 non-null   float64 
 5   DiscCH          1070 non-null   float64 
 6   DiscMM          1070 non-null   float64 
 7   SpecialCH       1070 non-null   int64   
 8   SpecialMM       1070 non-null   int64   
 9   LoyalCH         1070 non-null   float64 
 10  SalePriceMM     1070 non-null   float64 
 11  SalePriceCH     1070 non-null   float64 
 12  PriceDiff       1070 non-null   float64 
 13  Store7          1070 non-null   category
 14  PctDiscMM       1070 non-null   float64 
 15  PctDiscCH       1070 non-null   float64 
 16  ListPriceDiff   1070 non-null   float64 
 17  STORE         

In [5]:
oj.nunique()

Purchase            2
WeekofPurchase     52
StoreID             5
PriceCH            10
PriceMM             8
DiscCH             12
DiscMM             12
SpecialCH           2
SpecialMM           2
LoyalCH           553
SalePriceMM        18
SalePriceCH        13
PriceDiff          36
Store7              2
PctDiscMM          18
PctDiscCH          13
ListPriceDiff      18
STORE               5
dtype: int64